In [ ]:
# ============================================================
# NOTEBOOK 10
# VLM PARSER VALIDATION AND CORRECTED EVALUATION
# ============================================================
#
# Purpose:
#   1. Load the original zero-shot and fine-tuned predictions.
#   2. Diagnose parser failures.
#   3. Apply one shared normalization procedure to both models.
#   4. Compare predicted set-size distributions.
#   5. Recalculate Instrument, Action, and Tissue metrics.
#   6. Quantify the effect of normalization.
#
# IMPORTANT:
#   The original evaluation results are NOT overwritten.
#   All corrected outputs are saved separately.
# ============================================================

In [ ]:
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

print("Libraries loaded.")

Libraries loaded.


In [ ]:
# ============================================================
# PATHS
# ============================================================

RESULTS_DIR = Path(
    "/content/drive/MyDrive/Surgical-VLM/results"
)

ZERO_SHOT_PATH = (
    RESULTS_DIR / "ZeroShot Results/visionllm_zeroshot_predictions.csv"
)

FINETUNED_PATH = (
    RESULTS_DIR / "VLM Results/visionllm_predictions.csv"
)

CORRECTED_DIR = (
    RESULTS_DIR / "corrected_vlm_evaluation"
)

CORRECTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Results directory:", RESULTS_DIR)
print("Corrected output directory:", CORRECTED_DIR)

Results directory: /content/drive/MyDrive/Surgical-VLM/results
Corrected output directory: /content/drive/MyDrive/Surgical-VLM/results/corrected_vlm_evaluation


In [ ]:
# ============================================================
# LOAD ORIGINAL PREDICTIONS
# ============================================================

zero_shot_df = pd.read_csv(ZERO_SHOT_PATH)
finetuned_df = pd.read_csv(FINETUNED_PATH)

print("Zero-shot shape :", zero_shot_df.shape)
print("Fine-tuned shape:", finetuned_df.shape)

Zero-shot shape : (5600, 5)
Fine-tuned shape: (5600, 5)


In [ ]:
# ============================================================
# BASIC DATA INSPECTION
# ============================================================

print("\nZero-shot columns:")
print(zero_shot_df.columns.tolist())

print("\nFine-tuned columns:")
print(finetuned_df.columns.tolist())


Zero-shot columns:
['image_full_path', 'question', 'text', 'task', 'prediction']

Fine-tuned columns:
['image_full_path', 'task', 'question', 'ground_truth', 'prediction']


In [ ]:
# ============================================================
# TASK DISTRIBUTION
# ============================================================

print("\nZERO-SHOT")
print(
    zero_shot_df["task"]
    .value_counts()
    .sort_index()
)

print("\nFINE-TUNED")
print(
    finetuned_df["task"]
    .value_counts()
    .sort_index()
)


ZERO-SHOT
task
Action Recognition              800
Instrument Recognition          800
Phase Recognition               800
Safety Assessment               800
Surgical Image Captioning       800
Tissue and Organ Recognition    800
Triplet Recognition             800
Name: count, dtype: int64

FINE-TUNED
task
Action Recognition              800
Instrument Recognition          800
Phase Recognition               800
Safety Assessment               800
Surgical Image Captioning       800
Tissue and Organ Recognition    800
Triplet Recognition             800
Name: count, dtype: int64


In [ ]:
# ============================================================
# CHECK MISSING PREDICTIONS
# ============================================================

for name, df in [
    ("Zero-shot", zero_shot_df),
    ("Fine-tuned", finetuned_df)
]:

    missing = (
        df["prediction"]
        .isna()
        .sum()
    )

    empty = (
        df["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(
        f"{name}: "
        f"missing={missing}, "
        f"empty={empty}"
    )

Zero-shot: missing=0, empty=0
Fine-tuned: missing=0, empty=0


In [ ]:
# ============================================================
# CANONICAL LABEL VOCABULARIES
# ============================================================

CANONICAL_LABELS = {

    "Instrument Recognition": [
        "grasper",
        "bipolar",
        "hook",
        "scissors",
        "clipper",
        "irrigator",
    ],

    "Action Recognition": [
        "grasp",
        "retract",
        "dissect",
        "coagulate",
        "clip",
        "cut",
        "aspirate",
        "irrigate",
        "pack",
        "null_verb",
    ],

    "Tissue and Organ Recognition": [
        "gallbladder",
        "cystic artery",
        "cystic duct",
        "cystic plate",
        "liver",
        "specimen bag",
        "fluid",
        "abdominal wall cavity",
        "omentum",
        "blood_vessel",
        "gut",
        "peritoneum",
        "cystic_pedicle",
        "adhesion",
        "null_target",
    ],

    "Phase Recognition": [
        "Preparation",
        "Calot Triangle Dissection",
        "Clipping Cutting",
        "Gallbladder Dissection",
        "Gallbladder Packaging",
        "Cleaning Coagulation",
        "Gallbladder Retraction",
    ]
}

for task, labels in CANONICAL_LABELS.items():
    print(
        f"{task}: {len(labels)} labels"
    )

Instrument Recognition: 6 labels
Action Recognition: 10 labels
Tissue and Organ Recognition: 15 labels
Phase Recognition: 7 labels


In [ ]:
# ============================================================
# SAFE TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Treat underscores and hyphens as spaces
    text = text.replace("_", " ")
    text = text.replace("-", " ")

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
# ============================================================
# CANDIDATE SYNONYM MAP
#
# These mappings should only be retained if they are justified
# by the audit and by the task ontology.
# ============================================================

CANDIDATE_SYNONYMS = {

    "Instrument Recognition": {

        "shears": "scissors",
        "surgical shears": "scissors",
        "surgical scissors": "scissors",
        "laparoscopic scissors": "scissors",

        "clip applicator": "clipper",
        "clipping device": "clipper",

        "bipolar cautery": "bipolar",
        "bipolar cauterizer": "bipolar",
    },

    "Action Recognition": {

        "grasping": "grasp",
        "grasped": "grasp",

        "dissecting": "dissect",
        "dissection": "dissect",

        "retracting": "retract",
        "retraction": "retract",

        "cutting": "cut",

        "coagulation": "coagulate",
        "coagulating": "coagulate",
        "cauterization": "coagulate",

        "clipping": "clip",

        "aspiration": "aspirate",
        "aspirating": "aspirate",

        "irrigation": "irrigate",
        "irrigating": "irrigate",
        "flushing": "irrigate",

        "packing": "pack",
        "packed": "pack",
    },

    "Tissue and Organ Recognition": {},

    "Phase Recognition": {}
}

In [ ]:
# ============================================================
# PHRASE MATCHING
# ============================================================

def phrase_present(text, phrase):

    text = normalize_text(text)
    phrase = normalize_text(phrase)

    # Escape the phrase for regex
    pattern = re.escape(phrase)

    # Permit whitespace between words
    pattern = pattern.replace(
        r"\ ",
        r"\s+"
    )

    return bool(
        re.search(
            r"(?<!\w)" +
            pattern +
            r"(?!\w)",
            text
        )
    )

In [ ]:
# ============================================================
# SHARED PARSER
# ============================================================

def extract_labels(
    prediction,
    task,
    use_synonyms=False
):

    text = normalize_text(prediction)

    labels = CANONICAL_LABELS[task]

    found = []

    # --------------------------------------------------------
    # Exact canonical labels
    # --------------------------------------------------------

    for label in labels:

        if phrase_present(text, label):
            found.append(label)

    # --------------------------------------------------------
    # Optional synonym mappings
    # --------------------------------------------------------

    if use_synonyms:

        synonym_map = CANDIDATE_SYNONYMS.get(
            task,
            {}
        )

        for variant, canonical in synonym_map.items():

            if phrase_present(text, variant):
                found.append(canonical)

    # Remove duplicates
    found = list(dict.fromkeys(found))

    return found

In [ ]:
# ============================================================
# ORIGINAL / EXACT-LABEL PARSER FAILURE RATES
# ============================================================

def parser_summary(df, use_synonyms=False):

    rows = []

    for task in CANONICAL_LABELS:

        subset = df[
            df["task"] == task
        ].copy()

        extracted = subset.apply(
            lambda row:
            extract_labels(
                row["prediction"],
                row["task"],
                use_synonyms=use_synonyms
            ),
            axis=1
        )

        sizes = extracted.apply(len)

        rows.append({
            "Task": task,
            "N": len(subset),
            "Mean predicted set size": sizes.mean(),
            "Zero-label outputs": (sizes == 0).sum(),
            "Zero-label rate": (sizes == 0).mean()
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# ORIGINAL EXACT-MATCH PARSER DIAGNOSTIC
# ============================================================

zs_exact_summary = parser_summary(
    zero_shot_df,
    use_synonyms=False
)

ft_exact_summary = parser_summary(
    finetuned_df,
    use_synonyms=False
)

print("ZERO-SHOT")
display(zs_exact_summary)

print("\nFINE-TUNED")
display(ft_exact_summary)

ZERO-SHOT


,Task,N,Mean predicted set size,Zero-label outputs,Zero-label rate
0,Instrument Recognition,800,0.23250,615,0.76875
1,Action Recognition,800,0.43000,481,0.60125
2,Tissue and Organ Recognition,800,1.97250,0,0.00000
3,Phase Recognition,800,0.19125,647,0.80875



FINE-TUNED


,Task,N,Mean predicted set size,Zero-label outputs,Zero-label rate
0,Instrument Recognition,800,3.05500,1,0.00125
1,Action Recognition,800,2.33500,0,0.00000
2,Tissue and Organ Recognition,800,3.30375,0,0.00000
3,Phase Recognition,800,1.07375,0,0.00000


In [ ]:
# ============================================================
# EFFECT OF SYNONYM NORMALIZATION
# ============================================================

zs_syn_summary = parser_summary(
    zero_shot_df,
    use_synonyms=True
)

ft_syn_summary = parser_summary(
    finetuned_df,
    use_synonyms=True
)

comparison = zs_exact_summary.merge(
    zs_syn_summary,
    on=["Task", "N"],
    suffixes=(
        " exact",
        " normalized"
    )
)

comparison[
    [
        "Task",
        "N",
        "Mean predicted set size exact",
        "Mean predicted set size normalized",
        "Zero-label outputs exact",
        "Zero-label outputs normalized",
        "Zero-label rate exact",
        "Zero-label rate normalized"
    ]
]

,Task,N,Mean predicted set size exact,Mean predicted set size normalized,Zero-label outputs exact,Zero-label outputs normalized,Zero-label rate exact,Zero-label rate normalized
0,Instrument Recognition,800,0.23250,0.28500,615,573,0.76875,0.71625
1,Action Recognition,800,0.43000,0.95875,481,184,0.60125,0.23000
2,Tissue and Organ Recognition,800,1.97250,1.97250,0,0,0.00000,0.00000
3,Phase Recognition,800,0.19125,0.19125,647,647,0.80875,0.80875


In [ ]:
# ============================================================
# IDENTIFY PARSER FAILURES
# ============================================================

def add_extraction_columns(
    df,
    use_synonyms=False
):

    out = df.copy()

    # Only parse the recognition tasks that use
    # canonical label sets.
    recognition_tasks = set(
        CANONICAL_LABELS.keys()
    )

    def parse_row(row):

        task = row["task"]

        if task not in recognition_tasks:
            return []

        return extract_labels(
            row["prediction"],
            task,
            use_synonyms=use_synonyms
        )

    out["parsed_labels"] = out.apply(
        parse_row,
        axis=1
    )

    out["predicted_set_size"] = (
        out["parsed_labels"]
        .apply(len)
    )

    return out


zs_exact = add_extraction_columns(
    zero_shot_df,
    use_synonyms=False
)

zs_normalized = add_extraction_columns(
    zero_shot_df,
    use_synonyms=True
)

ft_normalized = add_extraction_columns(
    finetuned_df,
    use_synonyms=True
)

print("Extraction completed.")

Extraction completed.


In [ ]:
# ============================================================
# HOW MANY ZERO-SHOT FAILURES ARE RECOVERED?
# ============================================================

audit_summary = []

for task in [
    "Instrument Recognition",
    "Action Recognition",
    "Tissue and Organ Recognition"
]:

    exact = zs_exact[
        (zs_exact["task"] == task) &
        (zs_exact["predicted_set_size"] == 0)
    ]

    recovered = zs_normalized.loc[
        exact.index,
        "predicted_set_size"
    ].gt(0).sum()

    audit_summary.append({
        "Task": task,
        "Original parser failures": len(exact),
        "Recovered by normalization": recovered,
        "Recovery rate":
            recovered / len(exact)
            if len(exact) else np.nan
    })

audit_recovery_df = pd.DataFrame(
    audit_summary
)

display(audit_recovery_df)

,Task,Original parser failures,Recovered by normalization,Recovery rate
0,Instrument Recognition,615,42,0.068293
1,Action Recognition,481,297,0.617464
2,Tissue and Organ Recognition,0,0,NaN


In [ ]:
# ============================================================
# MEAN PREDICTED SET SIZE
# ============================================================

def set_size_table(df):

    rows = []

    for task in [
        "Instrument Recognition",
        "Action Recognition",
        "Tissue and Organ Recognition"
    ]:

        sub = df[
            df["task"] == task
        ]

        rows.append({
            "Task": task,
            "N": len(sub),
            "Mean predicted set size":
                sub["predicted_set_size"].mean(),
            "Median predicted set size":
                sub["predicted_set_size"].median(),
            "Std predicted set size":
                sub["predicted_set_size"].std(),
            "Zero-label rate":
                (
                    sub["predicted_set_size"] == 0
                ).mean()
        })

    return pd.DataFrame(rows)


zs_set_sizes = set_size_table(
    zs_normalized
)

ft_set_sizes = set_size_table(
    ft_normalized
)

set_size_comparison = zs_set_sizes.merge(
    ft_set_sizes,
    on=["Task", "N"],
    suffixes=(
        " zero-shot",
        " fine-tuned"
    )
)

display(set_size_comparison)

,Task,N,Mean predicted set size zero-shot,Median predicted set size zero-shot,Std predicted set size zero-shot,Zero-label rate zero-shot,Mean predicted set size fine-tuned,Median predicted set size fine-tuned,Std predicted set size fine-tuned,Zero-label rate fine-tuned
0,Instrument Recognition,800,0.28500,0.0,0.454459,0.71625,3.05500,2.0,1.757366,0.00125
1,Action Recognition,800,0.95875,1.0,0.687488,0.23000,2.33500,1.0,1.631288,0.00000
2,Tissue and Organ Recognition,800,1.97250,2.0,0.163637,0.00000,3.30375,3.0,0.758093,0.00000


In [ ]:
# ============================================================
# INSPECT POSSIBLE GROUND-TRUTH COLUMNS
# ============================================================

print(
    zero_shot_df.columns.tolist()
)

print("\nExample row:")
display(
    zero_shot_df.iloc[0]
)

['image_full_path', 'question', 'text', 'task', 'prediction']

Example row:


,0
image_full_path,/content/CholecT50/CholecT50/videos/VID110/001...
question,Identify the grasper's action in this surgery ...
text,The grasper is performing a retract action in ...
task,Action Recognition
prediction,"In the provided image, the grasper is being us..."


In [ ]:

# ============================================================
# GROUND-TRUTH COLUMNS
# ============================================================

ZERO_SHOT_GT_COLUMN = "text"
FINETUNED_GT_COLUMN = "ground_truth"

print("Zero-shot ground truth :", ZERO_SHOT_GT_COLUMN)
print("Fine-tuned ground truth:", FINETUNED_GT_COLUMN)

Zero-shot ground truth : text
Fine-tuned ground truth: ground_truth


In [ ]:
# ============================================================
# GROUND-TRUTH EXTRACTION FROM FULL SENTENCES
# ============================================================

def extract_ground_truth_labels(text, task):

    if task not in CANONICAL_LABELS:
        return []

    return extract_labels(
        text,
        task,
        use_synonyms=False
    )

In [ ]:
# ============================================================
# CONVERT GROUND TRUTH TO SET
# ============================================================

def to_label_set(value):

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return set(
            str(x).strip()
            for x in value
            if str(x).strip()
        )

    if pd.isna(value):
        return set()

    text = str(value).strip()

    # Handle stringified lists
    if text.startswith("[") and text.endswith("]"):

        try:
            parsed = json.loads(
                text.replace("'", '"')
            )

            if isinstance(parsed, list):
                return set(
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                )

        except Exception:
            pass

    # Handle common separators
    parts = re.split(
        r"\s*\|\s*|\s*;\s*|\s*,\s*",
        text
    )

    return set(
        p.strip()
        for p in parts
        if p.strip()
    )

In [ ]:
# ============================================================
# GROUND-TRUTH EXTRACTION AUDIT
# ============================================================

for task in [
    "Instrument Recognition",
    "Action Recognition",
    "Tissue and Organ Recognition"
]:

    print("\n" + "=" * 70)
    print(task)
    print("=" * 70)

    zs_examples = (
        zero_shot_df[
            zero_shot_df["task"] == task
        ]
        .head(10)
    )

    ft_examples = (
        finetuned_df[
            finetuned_df["task"] == task
        ]
        .head(10)
    )

    print("\nZERO-SHOT GROUND TRUTH")

    for _, row in zs_examples.iterrows():

        extracted = extract_ground_truth_labels(
            row[ZERO_SHOT_GT_COLUMN],
            task
        )

        print(
            f"\nRaw:       {row[ZERO_SHOT_GT_COLUMN]}"
            f"\nExtracted: {extracted}"
        )

    print("\nFINE-TUNED GROUND TRUTH")

    for _, row in ft_examples.iterrows():

        extracted = extract_ground_truth_labels(
            row[FINETUNED_GT_COLUMN],
            task
        )

        print(
            f"\nRaw:       {row[FINETUNED_GT_COLUMN]}"
            f"\nExtracted: {extracted}"
        )


Instrument Recognition

ZERO-SHOT GROUND TRUTH

Raw:       hook
Extracted: ['hook']

Raw:       bipolar
Extracted: ['bipolar']

Raw:       grasper
Extracted: ['grasper']

Raw:       ## Surgical Answers
(1) They fit into Clipper.
(2) The identified phase is Clipping Cutting.
Extracted: ['clipper']

Raw:       ## Surgical Answers
(1) They fall under the categories Grasper, Hook.
(2) The identified phase is Gallbladder Dissection.
Extracted: ['grasper', 'hook']

Raw:       grasper, hook
Extracted: ['grasper', 'hook']

Raw:       hook
Extracted: ['hook']

Raw:       They are classified into the categories: grasper, hook.
Extracted: ['grasper', 'hook']

Raw:       hook
Extracted: ['hook']

Raw:       hook
Extracted: ['hook']

FINE-TUNED GROUND TRUTH

Raw:       hook
Extracted: ['hook']

Raw:       bipolar
Extracted: ['bipolar']

Raw:       grasper
Extracted: ['grasper']

Raw:       ## Surgical Answers
(1) They fit into Clipper.
(2) The identified phase is Clipping Cutting.
Extracted: ['cli

In [ ]:
# ============================================================
# MULTI-LABEL EVALUATION
# ============================================================

def evaluate_multilabel(
    df,
    ground_truth_column,
    prediction_column="parsed_labels"
):

    rows = []

    for task in [
        "Instrument Recognition",
        "Action Recognition",
        "Tissue and Organ Recognition"
    ]:

        sub = df[
            df["task"] == task
        ].copy()

        y_true = []
        y_pred = []

        exact_matches = 0
        evaluated = 0

        labels = CANONICAL_LABELS[task]

        for _, row in sub.iterrows():

            true_set = set(
                extract_ground_truth_labels(
                    row[ground_truth_column],
                    task
                )
            )

            pred_set = set(
                row[prediction_column]
            )

            true_set &= set(labels)
            pred_set &= set(labels)

            if not true_set:
                continue

            evaluated += 1

            if true_set == pred_set:
                exact_matches += 1

            y_true.append([
                int(label in true_set)
                for label in labels
            ])

            y_pred.append([
                int(label in pred_set)
                for label in labels
            ])

        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        rows.append({
            "Task": task,
            "N": evaluated,
            "Mean GT set size":
                y_true.sum(axis=1).mean(),

            "Exact set accuracy":
                exact_matches / evaluated,

            "Micro precision":
                precision_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                ),

            "Micro recall":
                recall_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                ),

            "Micro F1":
                f1_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0
                )
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# EVALUATE BOTH MODELS WITH SHARED GROUND-TRUTH LOGIC
# ============================================================

zs_metrics = evaluate_multilabel(
    zs_normalized,
    ground_truth_column=ZERO_SHOT_GT_COLUMN
)

ft_metrics = evaluate_multilabel(
    ft_normalized,
    ground_truth_column=FINETUNED_GT_COLUMN
)

print("ZERO-SHOT")
display(zs_metrics)

print("\nFINE-TUNED")
display(ft_metrics)

ZERO-SHOT


,Task,N,Mean GT set size,Exact set accuracy,Micro precision,Micro recall,Micro F1
0,Instrument Recognition,799,1.44806,0.02378,0.167401,0.032844,0.054913
1,Action Recognition,800,1.17375,0.21875,0.443286,0.362087,0.398593
2,Tissue and Organ Recognition,800,1.19625,0.05250,0.456907,0.753396,0.568836



FINE-TUNED


,Task,N,Mean GT set size,Exact set accuracy,Micro precision,Micro recall,Micro F1
0,Instrument Recognition,799,1.44806,0.443054,0.451718,0.954192,0.613163
1,Action Recognition,800,1.17375,0.452500,0.456638,0.908413,0.607766
2,Tissue and Organ Recognition,800,1.19625,0.017500,0.354143,0.978056,0.520000


In [ ]:
# ============================================================
# PHASE ACCURACY
# ============================================================

def evaluate_phase(
    df,
    ground_truth_column=GROUND_TRUTH_COLUMN
):

    sub = df[
        df["task"] == "Phase Recognition"
    ].copy()

    correct = 0
    evaluated = 0

    labels = CANONICAL_LABELS[
        "Phase Recognition"
    ]

    for _, row in sub.iterrows():

        true_set = to_label_set(
            row[ground_truth_column]
        )

        pred_set = set(
            row["parsed_labels"]
        )

        true_set &= set(labels)
        pred_set &= set(labels)

        if len(true_set) != 1:
            continue

        evaluated += 1

        true_label = next(iter(true_set))

        if (
            len(pred_set) == 1 and
            next(iter(pred_set)) == true_label
        ):
            correct += 1

    return {
        "N": evaluated,
        "Accuracy": (
            correct / evaluated
            if evaluated else np.nan
        ),
        "Chance": 1 / len(labels)
    }


zs_phase = evaluate_phase(
    zs_normalized,
    ground_truth_column="text"
)

ft_phase = evaluate_phase(
    ft_normalized,
    ground_truth_column="ground_truth"
)

phase_results = pd.DataFrame([
    {
        "Model": "Zero-shot",
        **zs_phase
    },
    {
        "Model": "Fine-tuned",
        **ft_phase
    }
])

display(phase_results)

,Model,N,Accuracy,Chance
0,Zero-shot,526,0.000000,0.142857
1,Fine-tuned,526,0.750951,0.142857


In [ ]:
# ============================================================
# COMBINED MODEL COMPARISON
# ============================================================

comparison_rows = []

for task in [
    "Instrument Recognition",
    "Action Recognition",
    "Tissue and Organ Recognition"
]:

    zs_row = zs_metrics[
        zs_metrics["Task"] == task
    ].iloc[0]

    ft_row = ft_metrics[
        ft_metrics["Task"] == task
    ].iloc[0]

    comparison_rows.append({
        "Task": task,

        "Zero-shot F1":
            zs_row["Micro F1"],

        "Fine-tuned F1":
            ft_row["Micro F1"],

        "Absolute gain":
            ft_row["Micro F1"]
            - zs_row["Micro F1"],

        "Zero-shot precision":
            zs_row["Micro precision"],

        "Fine-tuned precision":
            ft_row["Micro precision"],

        "Zero-shot recall":
            zs_row["Micro recall"],

        "Fine-tuned recall":
            ft_row["Micro recall"],
    })

model_comparison_df = pd.DataFrame(
    comparison_rows
)

display(model_comparison_df)

,Task,Zero-shot F1,Fine-tuned F1,Absolute gain,Zero-shot precision,Fine-tuned precision,Zero-shot recall,Fine-tuned recall
0,Instrument Recognition,0.054913,0.613163,0.558250,0.167401,0.451718,0.032844,0.954192
1,Action Recognition,0.398593,0.607766,0.209173,0.443286,0.456638,0.362087,0.908413
2,Tissue and Organ Recognition,0.568836,0.520000,-0.048836,0.456907,0.354143,0.753396,0.978056


In [ ]:
# ============================================================
# PARSER EFFECT ON ZERO-SHOT RESULTS
# ============================================================

zs_original = add_extraction_columns(
    zero_shot_df,
    use_synonyms=False
)

zs_corrected = add_extraction_columns(
    zero_shot_df,
    use_synonyms=True
)

original_metrics = evaluate_multilabel(
    zs_original, "text"
)

corrected_metrics = evaluate_multilabel(
    zs_corrected, "text"
)

parser_effect = original_metrics[
    [
        "Task",
        "Micro precision",
        "Micro recall",
        "Micro F1"
    ]
].merge(
    corrected_metrics[
        [
            "Task",
            "Micro precision",
            "Micro recall",
            "Micro F1"
        ]
    ],
    on="Task",
    suffixes=(
        " original",
        " corrected"
    )
)

display(parser_effect)

,Task,Micro precision original,Micro recall original,Micro F1 original,Micro precision corrected,Micro recall corrected,Micro F1 corrected
0,Instrument Recognition,0.205405,0.032844,0.056632,0.167401,0.032844,0.054913
1,Action Recognition,0.415698,0.152290,0.222915,0.443286,0.362087,0.398593
2,Tissue and Organ Recognition,0.456907,0.753396,0.568836,0.456907,0.753396,0.568836


In [ ]:
# ============================================================
# PARSER EFFECT ON FINE-TUNED RESULTS
# ============================================================

ft_original = add_extraction_columns(
    finetuned_df,
    use_synonyms=False
)

ft_corrected = add_extraction_columns(
    finetuned_df,
    use_synonyms=True
)

ft_original_metrics = evaluate_multilabel(
    ft_original,
    ground_truth_column="ground_truth"
)

ft_corrected_metrics = evaluate_multilabel(
    ft_corrected,
    ground_truth_column="ground_truth"
)

ft_parser_effect = ft_original_metrics[
    [
        "Task",
        "Micro precision",
        "Micro recall",
        "Micro F1"
    ]
].merge(
    ft_corrected_metrics[
        [
            "Task",
            "Micro precision",
            "Micro recall",
            "Micro F1"
        ]
    ],
    on="Task",
    suffixes=(
        " original",
        " corrected"
    )
)

display(ft_parser_effect)

,Task,Micro precision original,Micro recall original,Micro F1 original,Micro precision corrected,Micro recall corrected,Micro F1 corrected
0,Instrument Recognition,0.451718,0.954192,0.613163,0.451718,0.954192,0.613163
1,Action Recognition,0.456638,0.908413,0.607766,0.456638,0.908413,0.607766
2,Tissue and Organ Recognition,0.354143,0.978056,0.520000,0.354143,0.978056,0.520000


In [ ]:
# ============================================================
# SAVE CORRECTED PREDICTIONS
# ============================================================

zs_save = zs_normalized.copy()
ft_save = ft_normalized.copy()

zs_save["parsed_labels"] = (
    zs_save["parsed_labels"]
    .apply(json.dumps)
)

ft_save["parsed_labels"] = (
    ft_save["parsed_labels"]
    .apply(json.dumps)
)

zs_output = (
    CORRECTED_DIR /
    "zero_shot_predictions_corrected.csv"
)

ft_output = (
    CORRECTED_DIR /
    "finetuned_predictions_corrected.csv"
)

zs_save.to_csv(
    zs_output,
    index=False
)

ft_save.to_csv(
    ft_output,
    index=False
)

print("Saved:")
print(zs_output)
print(ft_output)

Saved:
/content/drive/MyDrive/Surgical-VLM/results/corrected_vlm_evaluation/zero_shot_predictions_corrected.csv
/content/drive/MyDrive/Surgical-VLM/results/corrected_vlm_evaluation/finetuned_predictions_corrected.csv


In [ ]:
# ============================================================
# SAVE METRICS
# ============================================================

zs_metrics.to_csv(
    CORRECTED_DIR /
    "zero_shot_corrected_metrics.csv",
    index=False
)

ft_metrics.to_csv(
    CORRECTED_DIR /
    "finetuned_corrected_metrics.csv",
    index=False
)

model_comparison_df.to_csv(
    CORRECTED_DIR /
    "zero_shot_vs_finetuned_corrected.csv",
    index=False
)

set_size_comparison.to_csv(
    CORRECTED_DIR /
    "predicted_set_size_comparison.csv",
    index=False
)

audit_recovery_df.to_csv(
    CORRECTED_DIR /
    "parser_audit_recovery.csv",
    index=False
)

print("All corrected evaluation outputs saved.")

All corrected evaluation outputs saved.


In [ ]:
# ============================================================
# FINAL DIAGNOSTIC REPORT
# ============================================================

print("=" * 70)
print("VLM PARSER VALIDATION — FINAL REPORT")
print("=" * 70)

print("\n1. PARSER FAILURE RATES — ZERO SHOT")
print(
    zs_exact_summary[
        [
            "Task",
            "N",
            "Zero-label outputs",
            "Zero-label rate"
        ]
    ].to_string(index=False)
)

print("\n2. PARSER FAILURE RATES — FINE TUNED")
print(
    ft_exact_summary[
        [
            "Task",
            "N",
            "Zero-label outputs",
            "Zero-label rate"
        ]
    ].to_string(index=False)
)

print("\n3. NORMALIZED SET SIZES")
print(
    set_size_comparison.to_string(
        index=False
    )
)

print("\n4. CORRECTED ZERO-SHOT METRICS")
print(
    zs_metrics.to_string(
        index=False
    )
)

print("\n5. CORRECTED FINE-TUNED METRICS")
print(
    ft_metrics.to_string(
        index=False
    )
)

print("\n6. MODEL COMPARISON")
print(
    model_comparison_df.to_string(
        index=False
    )
)

print("=" * 70)

VLM PARSER VALIDATION — FINAL REPORT

1. PARSER FAILURE RATES — ZERO SHOT
                        Task   N  Zero-label outputs  Zero-label rate
      Instrument Recognition 800                 615          0.76875
          Action Recognition 800                 481          0.60125
Tissue and Organ Recognition 800                   0          0.00000
           Phase Recognition 800                 647          0.80875

2. PARSER FAILURE RATES — FINE TUNED
                        Task   N  Zero-label outputs  Zero-label rate
      Instrument Recognition 800                   1          0.00125
          Action Recognition 800                   0          0.00000
Tissue and Organ Recognition 800                   0          0.00000
           Phase Recognition 800                   0          0.00000

3. NORMALIZED SET SIZES
                        Task   N  Mean predicted set size zero-shot  Median predicted set size zero-shot  Std predicted set size zero-shot  Zero-label rate zero-sh